# Extraction Pipeline for existing Properties given from ArchiCAD

## 1. Loading and Exploring the IFC Model

In [1]:
# imports
import sys
import ifcopenshell
import glob
from pathlib import Path

# helpers
sys.path.insert(0, "../../")
from dataloader import _get_material_names, _clean_material_names
from geometric_extraction_helper import _IGNORE_TYPES, _SPLIT_PATTERN

/Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/.venv/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## 2. Material Analysis across all IFC models

In [2]:
# get all IFC file paths
ifc_paths = glob.glob("../../0_data/1_ifc_models/**/*.ifc", recursive=True)
print(f"Found {len(ifc_paths)} IFC files")

# collect all unique material names and all material combinations (as frozenset per element)
all_material_names = set()
all_combinations = set()

for path in ifc_paths:
    print(f"  Processing: {Path(path).name}")
    m = ifcopenshell.open(path)
    elements = [e for e in m.by_type("IfcProduct") if e.is_a() not in _IGNORE_TYPES]
    for element in elements:
        names = _get_material_names(element)
        if names:
            all_material_names.update(names)
            all_combinations.add(frozenset(names))

print(f"\nUnique materials : {len(all_material_names)}")
print(f"Unique combinations: {len(all_combinations)}")

Found 35 IFC files
  Processing: BRUT_41_ARC_Haus C_eBKPh.ifc
  Processing: BRUT_41_ARC_Haus B_eBKPh.ifc
  Processing: BRUT_41_ARC_Haus A_eBKPh.ifc
  Processing: ESAK_21_ARC_eBKPh.ifc
  Processing: LUMU_31_ARC_eBKPh_V3.ifc
  Processing: GERB_31_ARC_eBKPh_250221.ifc
  Processing: GERB_31_ARC_eBKPh_241209.ifc
  Processing: CHLI_51_ALE_Haus B_eBKP-H.ifc
  Processing: CHLI_51_AVM_Haus A.ifc
  Processing: CHLI_51_AVM_Haus B.ifc
  Processing: CHLI_51_ALE_Haus A_eBKP-H.ifc
  Processing: CHLI_51_ANG_Haus B.ifc
  Processing: CHLI_51_ANG_Haus A.ifc
  Processing: GSRH_31_ARC_eBKPh_V2.ifc
  Processing: GSRH_31_ARC_eBKPh_V1.ifc
  Processing: ADEM_41_ARC_eBKPh_250406.ifc
  Processing: ADEM_41_ARC_eBKPh_250701.ifc
  Processing: ADEM_41_ARC_eBKPh_260202.ifc
  Processing: IMBU_52_ARC_eBKPh_260127.ifc
  Processing: RALU_32_ARC_eBKPh.ifc
  Processing: ZUST_P_ARC_VOL_.ifc
  Processing: ZUST_P_ARC_EBK_.ifc
  Processing: ZUST_ARC_NGF_.ifc
  Processing: ZUST_A_ARC_NGF_.ifc
  Processing: ZUST_C_ARC_VOL_.ifc
 

In [3]:
# flat set of all cleaned individual material names
cleaned_materials: set[str] = set()
for combo in all_combinations:
    cleaned_materials.update(_clean_material_names(list(combo), _SPLIT_PATTERN))

# cleaned combinations
cleaned_combinations: set[frozenset] = {
    _clean_material_names(list(combo), _SPLIT_PATTERN)
    for combo in all_combinations
    if _clean_material_names(list(combo), _SPLIT_PATTERN)
}

for name in sorted(cleaned_materials):
    print(f"  {name}")

  allgemein
  alucobond
  aluminium
  ansicht
  anthrazit
  ausgerichtet
  aussen
  aussenfassade
  aussparung
  backstein
  balkondecken
  bekleidung
  belag
  bestand
  beton
  betonwerkstein
  bodenaufbau
  bodenbelag
  braun
  brüstung
  dachbekleidung
  dachrand
  deckendämmung
  drainagematte
  dämmeinlage
  dämmputz
  dämmung
  einbauten
  erdbereich
  fensterzarge
  flachdach
  foamglas
  gebäudegeometrie
  gebäudevolumen
  gedreht
  gedämmt
  gelb
  gips
  gipsplatte
  glas
  grau
  handlauf
  hellgrau
  holz
  holzwerkstoff
  horizontal
  imprägniert
  innen
  kalksandstein
  kellertrennwand
  keramik
  kern
  kies
  konstruktion
  kragplattenanschluss
  kunststein
  kunststoff
  luft
  magerbeton
  metall
  metallverkleidung
  mieterausbau
  mörtel
  naturstein
  neubau
  operator
  perimeter
  perimeterdämmung
  pflaster
  priorität
  putz
  schalldämmung
  schraffur
  schroppen
  schwächer
  sichtbeton
  sichtmauerwerk
  sickerplatte
  sperrschicht
  stahl
  stahlbeton
  s

In [4]:
# based on manual review, exclude certain words that are not actual materials
EXCLUDE_WORDS = {
    "weniger", "weit", "vertikal", "transparente", "schwächer",
    "schraffur", "innen", "horizontal", "grau", "gelb", "gedreht",
    "braun", "aussen", "ausgerichtet", "anthrazit", "operator", "kern"
}

cleaned_materials -= EXCLUDE_WORDS
cleaned_combinations = {combo - EXCLUDE_WORDS for combo in cleaned_combinations}

# remove combos that became empty after exclusion
cleaned_combinations = {c for c in cleaned_combinations if c}

print(f"Current list after manual removal: {len(cleaned_materials)}")

print("\nRemaining material names:")
for name in sorted(cleaned_materials):
    print(f"  {name}")

Current list after manual removal: 89

Remaining material names:
  allgemein
  alucobond
  aluminium
  ansicht
  aussenfassade
  aussparung
  backstein
  balkondecken
  bekleidung
  belag
  bestand
  beton
  betonwerkstein
  bodenaufbau
  bodenbelag
  brüstung
  dachbekleidung
  dachrand
  deckendämmung
  drainagematte
  dämmeinlage
  dämmputz
  dämmung
  einbauten
  erdbereich
  fensterzarge
  flachdach
  foamglas
  gebäudegeometrie
  gebäudevolumen
  gedämmt
  gips
  gipsplatte
  glas
  handlauf
  hellgrau
  holz
  holzwerkstoff
  imprägniert
  kalksandstein
  kellertrennwand
  keramik
  kies
  konstruktion
  kragplattenanschluss
  kunststein
  kunststoff
  luft
  magerbeton
  metall
  metallverkleidung
  mieterausbau
  mörtel
  naturstein
  neubau
  perimeter
  perimeterdämmung
  pflaster
  priorität
  putz
  schalldämmung
  schroppen
  sichtbeton
  sichtmauerwerk
  sickerplatte
  sperrschicht
  stahl
  stahlbeton
  steildach
  streckmetall
  substrat
  terazzo
  thermurelement
  tr